# 04 — Matched-filter library + trials correction

Iterate over the default exotic library, inject each template into a background spectrum, and confirm the matched template wins. Then demonstrate the Gross–Vitells / Bonferroni trials correction by null-running on background-only spectra.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from anomalymetric.ingest.synthetic import synthetic_natural
from anomalymetric.models.exotic import default_library
from anomalymetric.models.powerlaw import PowerLaw
from anomalymetric.models.thermal import BlackBody
from anomalymetric.score.loeb_turner import loeb_turner_score
from anomalymetric.score.trials import gross_vitells_global_p

def natural():
    return [BlackBody(T_K=300., amplitude=1.), PowerLaw(amplitude=1e-3, index=2., reference_eV=1.)]

lib = default_library()
print('library size:', len(lib))

library size: 12


In [2]:
# Null distribution: many background-only trials.
rng = np.random.default_rng(0)
ts_null = []
for i in range(30):
    spec = synthetic_natural(log_e_min=-2, log_e_max=4, bins_per_decade=20, exposure_cm2_s=1e5, seed=int(rng.integers(0, 10_000)))
    res = loeb_turner_score(spec, natural())
    ts_null.append(res.test_statistic)
ts_null = np.array(ts_null)
print('null TS percentiles:', np.percentile(ts_null, [50, 90, 95, 99]))

null TS percentiles: [1.13578465 3.75283871 3.94265623 4.46216965]


In [3]:
# Convert observed TS to corrected global p across the library.
for ts in [4, 9, 16, 25]:
    p_global = gross_vitells_global_p(ts, n_trials=len(lib))
    print(f'TS={ts}  p_global={p_global:.3g}  -log10 p={-np.log10(max(p_global, 1e-300)):.2f}')

TS=4  p_global=0.241  -log10 p=0.62
TS=9  p_global=0.0161  -log10 p=1.79
TS=16  p_global=0.00038  -log10 p=3.42
TS=25  p_global=3.44e-06  -log10 p=5.46
